In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
from ingest import load_faq_data

documents = load_faq_data()

In [3]:
texts = []

for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

In [4]:
from tqdm.auto import tqdm

In [5]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/25 [00:00<?, ?it/s]

1208

In [6]:
vectors[0]

array([-1.03578880e-01, -6.19631968e-02, -1.13746319e-02, -3.32001527e-03,
        2.98825316e-02, -1.73465889e-02, -9.54300687e-02,  3.11983228e-02,
       -1.53322026e-01,  7.39516765e-02,  4.03770991e-02, -4.62293066e-02,
        1.96901523e-02, -3.71884555e-02, -1.84310898e-02,  2.22527068e-02,
        5.88773657e-03, -5.50784245e-02,  5.64356931e-02, -1.05830438e-01,
        4.25289720e-02, -3.97675894e-02, -1.17406743e-02,  7.39043653e-02,
        4.40801382e-02,  5.91113865e-02,  6.24630041e-02, -2.63405498e-02,
       -8.60160030e-03,  5.82082607e-02, -1.20065603e-02, -1.86081789e-03,
        2.75744852e-02,  3.93735245e-02,  6.54971227e-02,  6.49856180e-02,
        1.75079964e-02, -4.75835986e-02, -7.16398731e-02, -6.99092895e-02,
       -6.10002466e-02, -1.18092131e-02,  4.61798743e-04,  4.62616198e-02,
        1.83203761e-02, -8.85390639e-02, -3.84004898e-02, -9.99949798e-02,
       -4.70628478e-02,  4.94977385e-02, -4.06512804e-02, -1.30986661e-01,
       -6.99002594e-02,  

In [7]:
import numpy as np
X = np.array(vectors)

In [ ]:
query = 'Can I still join the course after the start date?'
v_query = model.encode(query)

In [11]:
scores = [v_query.dot(X[i]) for i in range(len(X))]

In [13]:
top5 = np.argsort(scores)[-5:]
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.56009996
{'id': '068529125b', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I follow the course after it finishes?', 'answer': 'Yes, we will keep all the materials available, so you can follow the course at your own pace after it finishes.\n\nYou can also continue reviewing the homeworks and prepare for the next cohort. You can also start working on your final capstone project.'}

0.6536312
{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'The course has already started. Can I still join it?', 'answer': 'Yes, you can. Even though you missed the start date

In [14]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)

In [15]:
query = 'I just discovered the course. Can I still join it?'
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [17]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv('GEMINI_API_KEY'),
    base_url='https://generativelanguage.googleapis.com/v1beta/openai/'
)

In [18]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)


In [19]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [20]:
query = 'I just found out about the program, can I still sign up?'
assistant.rag(query)

'Yes, you can still join. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. \n\nFurthermore, you can just start learning and submitting homework (while the submission forms are open) without registering, as registration is not checked against any list.'

In [23]:

class RAGVector(RAGBase):
    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {'course': self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [24]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [25]:
vector_assistant.rag('the program has already begun, can I still sign up?')


'Yes, you can still join. However, if you want to receive a certificate, you must submit your project while submissions are still being accepted. \n\nYou can also just start learning and submitting homework (while the submission forms are open) without officially registering, as registration is not checked against any list.'

In [26]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors2.db'
)

In [27]:
vs_index.fit(vectors, documents)

In [31]:
query = 'I just discovered the course. Can I still join it?'
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [33]:
results = vs_index.search(
    query_vector,
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

In [34]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co

In [35]:
vs_index.close()